# Deep Past Initiative – Machine Translation (Training Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Main ideas:
- Use **ByT5** to handle noisy Akkadian transliterations at the character level
- Perform **simple sentence alignment** to increase training data
- Fine-tune using HuggingFace `Trainer`


Inference Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-infer).

In [1]:
!pip uninstall transformers
!pip install transformers==4.57.1

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Would remove:
    /usr/local/bin/transformers
    /usr/local/lib/python3.12/dist-packages/transformers-5.0.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/transformers/*
Proceed (Y/n)? Y
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 133.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0


In [2]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [3]:
!pip install evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.9 MB/s eta 0:00:00


In [4]:
import os
import gc
import re
import math
import unicodedata
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import GroupKFold
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate


In [5]:
import os
from datetime import datetime
import ipykernel
import requests
import json

class Config:
    # Akkadian transliteration contains a lot of noise and many unknown words, so
    # ByT5, which processes text at the character (byte) level rather than the word level, is the strongest choice.
    MODEL_NAME = "google/byt5-small"

    # ByT5 tends to produce longer token sequences, but 512 tokens is enough at the sentence level.
    MAX_LENGTH = 512

    BATCH_SIZE = 8       # Adjust depending on GPU memory (on a P100 you can usually go with 8–16).
    EPOCHS = 10
    LEARNING_RATE = 2e-4

    # Dynamically determine the output directory based on notebook path and timestamp
    try:
        connection_file = os.path.basename(ipykernel.get_connection_file())
        kernel_id = connection_file.split('-', 1)[1].split('.')[0]
        response = requests.get('http://172.28.0.12:9000/api/sessions')
        sessions = json.loads(response.text)
        notebook_path = "unknown_notebook.ipynb" # Default in case of failure
        for session in sessions:
            if session['kernel']['id'] == kernel_id:
                notebook_path = session['path']
                break
        notebook_dir = os.path.dirname(notebook_path)
        notebook_name = os.path.splitext(os.path.basename(notebook_path))[0]
    except Exception:
        notebook_dir = "." # Fallback to current working directory
        notebook_name = "colab_notebook_run" # Fallback notebook name

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    OUTPUT_DIR = os.path.join(notebook_dir, f"{notebook_name}_{timestamp}")

    # ============ CV (5-fold) ============
    CV_N_SPLITS = 5
    CV_SEED = 42
    USE_GROUP_KFOLD = True
    SAVE_EACH_FOLD_MODEL = True

    # Save fold artifacts under OUTPUT_DIR
    FOLD_OUTPUT_ROOT = f"{OUTPUT_DIR}/cv5"
    REPORT_CSV = "cv_results.csv"

    # ============ Normalization flags (Discussion Entry 678899) ============
    REMOVE_TRANSLATION_ANNOTATIONS = True
    TRANSLATION_DECIMALS_TO_FRACTIONS = True
    TRANSLATION_MONTH_ROMAN_TO_INT = True
    TRANSLATION_COLLAPSE_SLASH_ALTS = False  # optional; may be too aggressive

    TRANSLIT_OPTIONAL_H_VARIANTS = False     # optional: Ḫ->H, ḫ->h
    TRANSLIT_OPTIONAL_KUB_DOT = False        # optional: KÙ.B. -> KÙ.BABBAR


In [6]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything()

In [7]:
INPUT_DIR = "/content/gdrive/MyDrive/Kaggle/Deep_Past_Challenge/data"
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

In [8]:
print(f"Original Train Data: {len(train_df)} docs")

Original Train Data: 1561 docs


In [9]:
def _clean_translation_for_alignment(text: str) -> str:
    """Light cleanup to avoid sentence-splitting on annotation tokens (e.g., 'fem.').

    NOTE: Full normalization is done later in the preprocessing cell.
    """
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)

    # Remove frequent annotation tokens known to NOT appear in test (Entry 678899).
    text = re.sub(r"\b(fem\.|sing\.|pl\.|plural)\b", " ", text, flags=re.IGNORECASE)
    text = text.replace("(?)", " ")

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。

    NOTE:
      - Sentence splitting increases samples per original document.
      - To avoid leakage in CV, we keep `oare_id` as a grouping key.
    """
    aligned_data = []

    for _, row in df.iterrows():
        oare_id = row["oare_id"]
        src = str(row["transliteration"])
        tgt_raw = str(row["translation"])

        tgt = _clean_translation_for_alignment(tgt_raw)

        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", tgt) if t.strip()]

        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]

        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:  # Remove junk/noisy data.
                    aligned_data.append({"oare_id": oare_id, "transliteration": s, "translation": t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({"oare_id": oare_id, "transliteration": src, "translation": tgt_raw})

    return pd.DataFrame(aligned_data, columns=["oare_id", "transliteration", "translation"])


In [10]:
# Run data augmentation.
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences (Alignment applied)")

# Prepare 5-fold CV splits.
cv_df = train_expanded.reset_index(drop=True)
assert set(["oare_id", "transliteration", "translation"]).issubset(cv_df.columns)

groups = cv_df["oare_id"].values
X_idx = np.arange(len(cv_df))

splitter = GroupKFold(n_splits=Config.CV_N_SPLITS)
fold_splits = list(splitter.split(X_idx, y=None, groups=groups))

print(f"Prepared {len(fold_splits)} folds (GroupKFold by oare_id)")


Expanded Train Data: 1561 sentences (Alignment applied)
Prepared 5 folds (GroupKFold by oare_id)


In [11]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
# Notes:
# - This cell incorporates discussion recommendations, especially Entry: 678899.
# - Some items are marked as optional; flip Config flags as needed.

tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

PREFIX_AKK2EN = "translate Akkadian to English: "
PREFIX_EN2AKK = "translate English to Akkadian: "

_SUBSCRIPT_NUM_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
})

# Entry 678899: decimal->unicode fraction suggestions
_DECIMAL_TO_FRACTION = {
    "0.5": "½",
    "0.25": "¼",
    "0.3333": "⅓",
    "0.8333": "⅚",
    "0.625": "⅝",
    "0.6666": "⅔",
    "0.75": "¾",
    "0.1666": "⅙",
}

_ROMAN_TO_INT = {
    "I": 1,
    "II": 2,
    "III": 3,
    "IV": 4,
    "V": 5,
    "VI": 6,
    "VII": 7,
    "VIII": 8,
    "IX": 9,
    "X": 10,
    "XI": 11,
    "XII": 12,
}


def _replace_decimals_to_fractions(text: str) -> str:
    # Replace exact recommended tokens; avoid touching arbitrary floats.
    for dec, frac in _DECIMAL_TO_FRACTION.items():
        text = re.sub(rf"(?<![0-9.]){re.escape(dec)}(?![0-9.])", frac, text)
    return text


def normalize_transliteration(text: str) -> str:
    """Normalize Akkadian transliteration towards test-style conventions.

    Key points (Entry 678899):
      - unify all gap markers into a single <gap>
      - align determinatives (e.g., (d)->{d})
      - unicode subscripts -> normal digits
      - (optional) unicode normalization tweaks
    """
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(_SUBSCRIPT_NUM_MAP)

    # Optional tweaks mentioned in the discussion
    if getattr(Config, "TRANSLIT_OPTIONAL_H_VARIANTS", False):
        text = text.replace("Ḫ", "H").replace("ḫ", "h")

    if getattr(Config, "TRANSLIT_OPTIONAL_KUB_DOT", False):
        # Example in Entry 678899: KÙ.B. -> KÙ.BABBAR
        text = text.replace("KÙ.B.", "KÙ.BABBAR")

    # Gap normalization (Entry 678899)
    text = text.replace("…", " <gap> ")
    text = re.sub(r"\(\s*large\s+break\s*\)", " <gap> ", text, flags=re.IGNORECASE)
    text = re.sub(r"\(\s*break\s*\)", " <gap> ", text, flags=re.IGNORECASE)
    text = re.sub(r"\(\s*n\s+broken\s+lines\s*\)", " <gap> ", text, flags=re.IGNORECASE)

    # [x] style gaps
    text = re.sub(r"\[\s*[xX](?:\s+[xX])*\s*\]", " <gap> ", text)
    # standalone x / xx / XXX tokens
    text = re.sub(r"\b[xX]{1,}\b", " <gap> ", text)

    # Deduplicate sequential gaps like: <gap> <gap>, <gap>-<gap>, etc.
    text = re.sub(r"(<gap>)(?:\s*[-–]?\s*<gap>)+", r"\1", text)

    # Remove leading dash directly before <gap> when it's a standalone marker (e.g., ' -<gap>')
    text = re.sub(r"(^|\s)-(?=<gap>)", r"\1", text)
    # Remove trailing dash directly after <gap> when it's a standalone marker (e.g., '<gap>- ')
    text = re.sub(r"(?<=<gap>)-(?=\s|$)", "", text)

    # Determinatives alignment (Entry 678899)
    text = re.sub(r"\(d\)", "{d}", text)
    text = re.sub(r"\(ki\)", "{ki}", text, flags=re.IGNORECASE)
    text = text.replace("(TÚG)", "TÚG")

    # Collapse whitespace noise introduced by normalization/replacements.
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_translation(text: str) -> str:
    """Normalize English translation towards test-style conventions.

    Based on host recommendations (Entry 678899). This is intentionally conservative:
    we apply highly specific replacements and avoid aggressive punctuation removal.
    """
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)

    # Remove known annotation tokens (not in test per host comment)
    if getattr(Config, "REMOVE_TRANSLATION_ANNOTATIONS", True):
        text = re.sub(r"\b(fem\.|sing\.|pl\.|plural)\b", " ", text, flags=re.IGNORECASE)
        text = text.replace("(?)", " ")

    # Replace literal PN token with <gap>
    text = re.sub(r"\bPN\b", "<gap>", text)

    # Replace special glossary tokens (Entry 678899)
    text = re.sub(r"(^|\s)-gold\b", r"\1pašallum gold", text)
    text = re.sub(r"(^|\s)-tax\b", r"\1šadduātum tax", text)
    text = re.sub(r"(^|\s)-textiles\b", r"\1kutānum textiles", text)

    # Specific numeric phrase replacements (Entry 678899)
    text = re.sub(r"\b1\s*/\s*12\s*\(shekel\)\b", "15 grains", text, flags=re.IGNORECASE)
    text = re.sub(r"\b5\s*/\s*12\s*shekel\b", "⅔ shekel 15 grains", text, flags=re.IGNORECASE)
    text = re.sub(r"\b5\s+11\s*/\s*12\s*shekels\b", "6 shekels less 15 grains", text, flags=re.IGNORECASE)
    text = re.sub(r"\b7\s*/\s*12\s*shekel\b", "½ shekel 15 grains", text, flags=re.IGNORECASE)

    # Convert selected decimals to unicode fractions
    if getattr(Config, "TRANSLATION_DECIMALS_TO_FRACTIONS", True):
        # Shorten a couple of common float artifacts mentioned by host
        text = text.replace("1.3333300000000001", "1.3333")
        text = text.replace("2.6666600000000003", "2.6666")
        text = _replace_decimals_to_fractions(text)

    # Roman numeral months -> integers (Entry 678899)
    if getattr(Config, "TRANSLATION_MONTH_ROMAN_TO_INT", True):
        def _month_repl(m):
            roman = m.group(1).upper()
            val = _ROMAN_TO_INT.get(roman)
            return f"month {val}" if val is not None else m.group(0)

        text = re.sub(r"\bmonth\s+([ivx]{1,4})\b", _month_repl, text, flags=re.IGNORECASE)

    # Optional: collapse short slash alternatives like "you / she" -> "you"
    if getattr(Config, "TRANSLATION_COLLAPSE_SLASH_ALTS", False):
        text = re.sub(
            r"\b([A-Za-z][A-Za-z'\-]*(?:\s+[A-Za-z][A-Za-z'\-]*){0,3})\s*/\s*([A-Za-z][A-Za-z'\-]*(?:\s+[A-Za-z][A-Za-z'\-]*){0,3})\b",
            r"\1",
            text,
        )

    # Remove a few stray-mark patterns (keep meaningful ?/! and quotes)
    text = re.sub(r"\bxx?\b", " ", text)  # remove standalone x/xx tokens
    text = text.replace("<<", " ").replace(">>", " ")
    text = re.sub(r"\.(?:\s*\.)+", ".", text)  # '..' -> '.'

    # Remove stray angle brackets except <gap>
    text = re.sub(r"<(?!gap>)([^>]*)>", r"\1", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_akk2en_input(transliteration: str) -> str:
    return PREFIX_AKK2EN + normalize_transliteration(transliteration)


def build_en2akk_input(translation: str) -> str:
    # normalize input a bit for stability in the reverse direction
    return PREFIX_EN2AKK + normalize_translation(translation)


def create_bidirectional_dataset(ds_split, seed: int = 42):
    """Make a 2x dataset: (akk->en) + (en->akk).

    This follows the idea in `notebooks/004/dpc-baseline-train-infer.ipynb`:
    keep task-direction as a natural-language prefix in the input.
    """
    df = ds_split.to_pandas()

    # Direction 1: Akkadian -> English (main task)
    df_fwd = df.copy()
    df_fwd["input_text"] = [build_akk2en_input(x) for x in df_fwd["transliteration"].tolist()]
    df_fwd["target_text"] = [normalize_translation(x) for x in df_fwd["translation"].tolist()]

    # Direction 2: English -> Akkadian (reverse task)
    df_bwd = df.copy()
    df_bwd["input_text"] = [build_en2akk_input(x) for x in df_bwd["translation"].tolist()]
    df_bwd["target_text"] = [normalize_transliteration(x) for x in df_bwd["transliteration"].tolist()]

    df_combined = pd.concat([df_fwd, df_bwd], ignore_index=True)
    df_combined = df_combined.sample(frac=1, random_state=seed).reset_index(drop=True)

    return Dataset.from_pandas(df_combined)


def create_unidirectional_dataset(ds_split):
    """Validation dataset is kept as Akkadian -> English only for evaluation."""
    df = ds_split.to_pandas()
    df["input_text"] = [build_akk2en_input(x) for x in df["transliteration"].tolist()]
    df["target_text"] = [normalize_translation(x) for x in df["translation"].tolist()]
    return Dataset.from_pandas(df)


def preprocess_function(examples):
    inputs = [str(ex) for ex in examples["input_text"]]
    targets = [str(ex) for ex in examples["target_text"]]

    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=Config.MAX_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [12]:
# ==========================================
# 4. 5-fold CV training (fine-tuning)
# ==========================================

# Metrics: competition uses geo mean of BLEU and chrF++ (corpus-level)
chrf_metric = evaluate.load("chrf")
bleu_metric = evaluate.load("sacrebleu")


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Sometimes Seq2SeqTrainer can return logits; convert them to token ids.
    if hasattr(preds, "ndim") and preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    # ByT5 decoding can crash if ids are negative/out of vocab.
    preds = np.asarray(preds)
    preds = preds.astype(np.int64, copy=False)
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [normalize_translation(p) for p in decoded_preds]
    decoded_labels = [normalize_translation(l) for l in decoded_labels]

    refs = [[x] for x in decoded_labels]
    chrf = chrf_metric.compute(predictions=decoded_preds, references=refs)["score"]
    bleu = bleu_metric.compute(predictions=decoded_preds, references=refs)["score"]

    geo_mean = (bleu * chrf) ** 0.5 if (bleu > 0 and chrf > 0) else 0.0

    return {"chrf": chrf, "bleu": bleu, "geo_mean": geo_mean}


# Build a single HF dataset and slice by fold indices.
ds_all = Dataset.from_pandas(cv_df)

fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    print("" + "=" * 60)
    print(f"FOLD {fold}/{Config.CV_N_SPLITS - 1}")
    print("=" * 60)

    ds_train_raw = ds_all.select(tr_idx.tolist())
    ds_val_raw = ds_all.select(va_idx.tolist())

    # Train is bidirectional (akk->en + en->akk): 2x samples
    ds_train = create_bidirectional_dataset(ds_train_raw, seed=int(Config.CV_SEED) + int(fold))
    # Validation stays unidirectional (akk->en) for metric consistency
    ds_val = create_unidirectional_dataset(ds_val_raw)

    # Tokenize inside fold
    tokenized_train = ds_train.map(
        preprocess_function,
        batched=True,
        remove_columns=ds_train.column_names,
    )
    tokenized_val = ds_val.map(
        preprocess_function,
        batched=True,
        remove_columns=ds_val.column_names,
    )

    # Fresh model per fold
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    fold_out = f"{Config.FOLD_OUTPUT_ROOT}/fold_{fold}"
    fold_model_out = f"{fold_out}/model"
    os.makedirs(fold_out, exist_ok=True)

    args = Seq2SeqTrainingArguments(
        output_dir=fold_out,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=Config.LEARNING_RATE,

        # === Key fixes ===
        fp16=False,                    # FP16 can be unstable for some setups; keep FP32 by default.
        bf16=True,              # A100ならまずこれを試す
        tf32=True,              # 速度目的（メモリはほぼ変わらない）
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        generation_max_length=Config.MAX_LENGTH,  # Crucial for ByT5; avoid too-short defaults
        generation_num_beams=2,
        # ==================

        weight_decay=0.01,
        num_train_epochs=Config.EPOCHS,
        predict_with_generate=True,
        logging_strategy="epoch",
        logging_steps=10,
        disable_tqdm=True,
        report_to="none",
        load_best_model_at_end=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    print("Starting Training (fold CV, FP32 mode)...")
    trainer.train()

    eval_metrics = trainer.evaluate()
    fold_result = {
        "fold": int(fold),
        "n_train": int(len(tokenized_train)),
        "n_val": int(len(tokenized_val)),
        "geo_mean": float(eval_metrics.get("eval_geo_mean", float("nan"))),
        "bleu": float(eval_metrics.get("eval_bleu", float("nan"))),
        "chrf": float(eval_metrics.get("eval_chrf", float("nan"))),
        "fold_model_path": fold_model_out,
    }
    fold_metrics.append(fold_result)

    print("Fold metrics:", fold_result)

    if Config.SAVE_EACH_FOLD_MODEL:
        trainer.save_model(fold_model_out)
        tokenizer.save_pretrained(fold_model_out)
        print(f"Saved fold model to: {fold_model_out}")

    # Explicit cleanup to avoid memory/disk pressure across folds
    del trainer, model, data_collator, tokenized_train, tokenized_val, ds_train, ds_val
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Summarize CV
cv_results = pd.DataFrame(fold_metrics)
report_path = f"{Config.FOLD_OUTPUT_ROOT}/{Config.REPORT_CSV}"
os.makedirs(Config.FOLD_OUTPUT_ROOT, exist_ok=True)
cv_results.to_csv(report_path, index=False)

print("" + "=" * 60)
print("5-FOLD CV SUMMARY")
print("=" * 60)
print(cv_results[["fold", "n_train", "n_val", "geo_mean", "bleu", "chrf", "fold_model_path"]])

mean_score = cv_results["geo_mean"].mean()
std_score = cv_results["geo_mean"].std(ddof=1)
print(f"geo_mean: mean={mean_score:.4f}, std={std_score:.4f}")

best_i = int(cv_results["geo_mean"].idxmax())
print(f"Best fold index: {best_i}")
print(f"Best fold model path: {cv_results.loc[best_i, 'fold_model_path']}")
print(f"Saved CV report CSV to: {report_path}")


FOLD 0/4


Map:   0%|          | 0/2496 [00:00<?, ? examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

...

Starting Training (fold CV, FP32 mode)...
{'loss': 1.8861, 'grad_norm': 1.1222858428955078, 'learning_rate': 0.00018025316455696203, 'epoch': 1.0}
{'eval_loss': 0.8988145589828491, 'eval_chrf': 10.040070375467504, 'eval_bleu': 0.4960859091982033, 'eval_geo_mean': 2.231756581849316, 'eval_runtime': 163.9908, 'eval_samples_per_second': 1.903, 'eval_steps_per_second': 0.122, 'epoch': 1.0}
{'loss': 1.0464, 'grad_norm': 1.0342590808868408, 'learning_rate': 0.00016025316455696204, 'epoch': 2.0}
{'eval_loss': 0.7672368288040161, 'eval_chrf': 18.18355226265784, 'eval_bleu': 3.155848674829418, 'eval_geo_mean': 7.5752583660097175, 'eval_runtime': 162.5871, 'eval_samples_per_second': 1.919, 'eval_steps_per_second': 0.123, 'epoch': 2.0}
{'loss': 0.9304, 'grad_norm': 3.3532533645629883, 'learning_rate': 0.00014025316455696204, 'epoch': 3.0}
{'eval_loss': 0.7190187573432922, 'eval_chrf': 23.15598867340535, 'eval_bleu': 5.040659611271862, 'eval_geo_mean': 10.803770492985448, 'eval_runtime': 160.9033,

Map:   0%|          | 0/2498 [00:00<?, ? examples/s]

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

/tmp/ipykernel_1984/733303929.py:111: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting Training (fold CV, FP32 mode)...
{'loss': 1.9603, 'grad_norm': 1.880250096321106, 'learning_rate': 0.00018025316455696203, 'epoch': 1.0}
{'eval_loss': 0.9320527911186218, 'eval_chrf': 9.724680318368453, 'eval_bleu': 0.5042460688642586, 'eval_geo_mean': 2.214414555023273, 'eval_runtime': 163.2625, 'eval_samples_per_second': 1.911, 'eval_steps_per_second': 0.123, 'epoch': 1.0}
{'loss': 1.0531, 'grad_norm': 1.1232857704162598, 'learning_rate': 0.00016025316455696204, 'epoch': 2.0}
{'eval_loss': 0.7834795713424683, 'eval_chrf': 17.666934297215743, 'eval_bleu': 3.0612248577325283, 'eval_geo_mean': 7.354077673683097, 'eval_runtime': 162.9252, 'eval_samples_per_second': 1.915, 'eval_steps_per_second': 0.123, 'epoch': 2.0}
{'loss': 0.9221, 'grad_norm': 0.9478996992111206, 'learning_rate': 0.00014025316455696204, 'epoch': 3.0}
{'eval_loss': 0.7253004312515259, 'eval_chrf': 22.735276124998435, 'eval_bleu': 5.035458388368448, 'eval_geo_mean': 10.699651250180834, 'eval_runtime': 161.9594,

In [13]:
# --- Notes ---
# Fold models are saved under:
#   ./byt5-akkadian-model/cv5/fold_{k}/model
# The CV summary is saved to:
#   ./byt5-akkadian-model/cv5/cv_results.csv
